In [0]:
# =====================================================
# 1️⃣ Widgets
# =====================================================

import json
from pyspark.sql.functions import current_timestamp

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

dbutils.widgets.text(
    "table_metadata",
    "{'table_id': '1', 'table_name': 'customers', 'source_system': 'sqlserver', "
    "'source_schema': 'banking', 'source_table': 'customers', "
    "'source_path': '', 'bronze_schema': 'bronze', "
    "'silver_schema': 'silver', 'active_flag': 'True', "
    "'load_order': '1', 'created_at': '2026-02-18 13:23:37.053711'}"
)

dbutils.widgets.text(
    "table_parameters",
    "{'load_type': 'MERGE', "
    "'primary_key': 'customer_id', "
    "'watermark_column': 'updated_at'}"
)

dbutils.widgets.text(
    "run_id",
    "472519629468310"
)
run_id=dbutils.widgets.get("run_id")

# Parse JSON safely
table_metadata = json.loads(dbutils.widgets.get("table_metadata").replace("'", '"'))
table_parameters = json.loads(dbutils.widgets.get("table_parameters").replace("'", '"'))

print("Table Metadata:", table_metadata)
print("Table Parameters:", table_parameters)
print(f"Run ID: {run_id}")

Table Metadata: {'table_id': '1', 'table_name': 'customers', 'source_system': 'sqlserver', 'source_schema': 'banking', 'source_table': 'customers', 'source_path': '', 'bronze_schema': 'bronze', 'silver_schema': 'silver', 'active_flag': 'True', 'load_order': '1', 'created_at': '2026-02-18 13:23:37.053711'}
Table Parameters: {'load_type': 'MERGE', 'primary_key': 'customer_id', 'watermark_column': 'updated_at'}
Run ID: 472519629468310


In [0]:
# =====================================================
# 2️⃣ Extract Variables
# =====================================================

table_id = int(table_metadata["table_id"])
table_name = table_metadata["table_name"]
source_system = table_metadata["source_system"].lower()
source_schema = table_metadata["source_schema"]
source_table = table_metadata["source_table"]
source_path = table_metadata["source_path"]
bronze_schema = table_metadata["bronze_schema"]

load_type = table_parameters.get("load_type")
watermark_column = table_parameters.get("watermark_column")

bronze_table_fqn = f"banking.{bronze_schema}.{table_name}"

print(f"Target Bronze Table: {bronze_table_fqn}")

Target Bronze Table: banking.bronze.customers


In [0]:
# =====================================================
# Make and entry to audit table
# =====================================================
entry_exists = spark.sql(f"""
    SELECT 1
    FROM banking.metadata.pipeline_runs
    WHERE run_id = {run_id} AND table_id = {table_id}
""").count() > 0

if entry_exists:
    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET
            layer = 'Silver',
            start_time = TIMESTAMP('{start_time}'),
            end_time = NULL,
            status = 'INPROGRESS',
            number_of_records = NULL,
            error_message = NULL
        WHERE run_id = {run_id} AND table_id = {table_id}
    """)
else:
    spark.sql(f"""
        INSERT INTO banking.metadata.pipeline_runs
        VALUES (
            {run_id},
            {table_id},
            'Silver',
            TIMESTAMP('{start_time}'),
            NULL,  -- end time
            'INPROGRESS',
            NULL, --number of records
            NULL -- error message
        )
    """)


In [0]:
# =====================================================
# 3️⃣ Get Last Watermark (For Filtering Only)
# =====================================================

last_watermark = None

if load_type in ["APPEND", "MERGE"] and watermark_column:
    watermark_df = spark.sql(f"""
        SELECT last_watermark_value
        FROM banking.metadata.table_watermarks
        WHERE table_id = {table_id}
    """)
    
    if watermark_df.count() > 0:
        last_watermark = watermark_df.first()["last_watermark_value"]

print("Last Watermark:", last_watermark)

Last Watermark: 1900-01-01 00:00:00


In [0]:
%sql
create schema if not exists banking.bronze

In [0]:
# =====================================================
# 4️⃣ Read Source
# =====================================================
#
# NOTE: All source data now lives inside Databricks:
#   - 'sqlserver' sources are Databricks Delta tables in banking.source
#   - 'blob'     sources are .csv files in the UC volume /Volumes/banking/source/raw_data
# The source_system values and the branch logic below are kept as-is.

from pyspark.sql.functions import col

# UC volume that holds the raw .csv files (blob sources)
RAW_DATA_VOLUME = "/Volumes/banking/source/raw_data"

try:
    if source_system == "sqlserver":

        # 🔐 Read connection JSON from secret
        # ❌ NOT REQUIRED — source data now lives in Databricks Delta tables
        #    (banking.source.<source_table>); no SQL Server / JDBC connection needed.
        #
        # secret_json = dbutils.secrets.get(
        #     scope="banking-scope",
        #     key="sqlserver-connection-json"
        # )
        #
        # config = json.loads(secret_json)
        #
        # jdbc_url = f"jdbc:sqlserver://{config['host']}:{config['port']};database={config['database']}"
        #
        # jdbc_properties = {
        #     "user": config["user"],
        #     "password": config["password"],
        #     "driver": config["driver"]
        # }
        #
        # # Build query
        # if load_type in ["APPEND", "MERGE"] and last_watermark:
        #     query = f"""
        #     (SELECT * FROM {source_schema}.{source_table}
        #      WHERE {watermark_column} > '{last_watermark}') AS src
        #     """
        # else:
        #     query = f"(SELECT * FROM {source_schema}.{source_table}) AS src"
        #
        # source_df = spark.read.jdbc(
        #     url=jdbc_url,
        #     table=query,
        #     properties=jdbc_properties
        # )

        # ✅ Read from the Databricks Delta table: banking.source.<source_table>
        source_delta_table = f"banking.source.{source_table}"

        source_df = spark.read.table(source_delta_table)

        # Apply the same watermark filter the JDBC query used to apply
        if load_type in ["APPEND", "MERGE"] and last_watermark:
            source_df = source_df.filter(col(watermark_column) > last_watermark)

    elif source_system == "blob":

        # ✅ Read .csv files from the UC volume /Volumes/banking/source/raw_data
        # pathGlobFilter ensures only *.csv files are picked up — any
        # non-csv files in the volume (e.g. .sql scripts) are ignored.
        blob_source_path = source_path if source_path else RAW_DATA_VOLUME

        source_df = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("pathGlobFilter", "*.csv")
            .option(
                "cloudFiles.schemaLocation",
                f"{RAW_DATA_VOLUME}/_schema/{table_name}"
            )
            .option("header", "true")
            .load(blob_source_path)
        )

    else:
        raise ValueError("Unsupported source_system")

    # =====================================================
    # 5️⃣ Add insert_timestamp
    # =====================================================

    source_df = source_df.withColumn("insert_timestamp", current_timestamp())

    # =====================================================
    # 6️⃣ Write to Bronze (Append Only)
    # =====================================================

    if source_system == "blob":

        (
            source_df.writeStream
            .format("delta")
            .option(
                "checkpointLocation",
                f"{RAW_DATA_VOLUME}/_checkpoints/{table_name}"
            )            
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(bronze_table_fqn)
        )

        records_read = None  # Streaming, can't count records here

    else:

        # The source Delta tables define column DEFAULTs (e.g. kyc_status,
        # created_at, updated_at). A plain saveAsTable() would try to copy those
        # DEFAULTs into the new bronze table and fail with
        # WRONG_COLUMN_DEFAULTS_FOR_DELTA_FEATURE_NOT_ENABLED.
        # → Pre-create the bronze table via CTAS (schema only, no DEFAULTs),
        #   then append into it.
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {bronze_table_fqn}
            TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
            AS SELECT
                *,
                CAST(NULL AS TIMESTAMP) AS insert_timestamp
            FROM banking.source.{source_table}
            WHERE 1 = 0
        """)

        # FULL loads carry a complete snapshot of the source, so bronze is
        # rebuilt (overwrite) instead of appended to — otherwise every pipeline
        # run stacks another copy of the same rows on top of the previous load.
        if load_type == "FULL":
            (
                source_df.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(bronze_table_fqn)
            )
        else:
            (
                source_df.write
                .format("delta")
                .mode("append")
                .saveAsTable(bronze_table_fqn)
            )

        records_read = source_df.count()

    print("Source → Bronze Load Completed Successfully.")
    print("Watermark will be updated after Silver load.")

except Exception as e:
    end_time = spark.sql("SELECT current_timestamp()").collect()[0][0]
    error_message = str(e)

    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET
            end_time = TIMESTAMP('{end_time}'),
            status = 'FAILED',
            error_message = {'NULL' if not error_message else "'" + error_message.replace("'", "") + "'"}
        WHERE table_id = {table_id} AND run_id = {run_id} 
    """)
    raise

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6029109309427430>, line 16
     15 try:
---> 16     if source_system == "sqlserver":
     17 
     18         # 🔐 Read connection JSON from secret
     19         # ❌ NOT REQUIRED — source data now lives in Databricks Delta tables
     20         #    (banking.source.<source_table>); no SQL Server / JDBC connection needed.
     21         #
     22         # secret_json = dbutils.secrets.get(
     23         #     scope="banking-scope",
     24         #     key="sqlserver-connection-json"
     25         # )
     26         #
     27         # config = json.loads(secret_json)
     28         #
     29         # jdbc_url = f"jdbc:sqlserver://{config['host']}:{config['port']};database={config['database']}"
     30         #
     31         # jdbc_properties = {
     32         #     "user": config["user"],
     33         #